In [1]:
from google.colab import drive
drive.mount('/content/drive')

Mounted at /content/drive


In [2]:
!pip install -q transformers accelerate sentencepiece pandas wandb huggingface_hub bitsandbytes

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 60.7/60.7 MB 13.3 MB/s eta 0:00:00


In [3]:
import json
import re
import pandas as pd
import torch
import wandb

from pathlib import Path
from huggingface_hub import login
from transformers import (
    AutoTokenizer,
    AutoModelForCausalLM,
    BitsAndBytesConfig
)

In [ ]:
from huggingface_hub import login
login("<HF_TOKEN>")

wandb.login(key="<WANDB_API_KEY>")

/usr/local/lib/python3.12/dist-packages/notebook/notebookapp.py:191: SyntaxWarning: invalid escape sequence '\/'
  | |_| | '_ \/ _` / _` |  _/ -_)
wandb: WARNING If you're specifying your api key in code, ensure this code is not shared publicly.
wandb: WARNING Consider setting the WANDB_API_KEY environment variable, or running `wandb login` from the command line.
wandb: [wandb.login()] Using explicit session credentials for https://api.wandb.ai.
wandb: No netrc file found, creating one.
wandb: Appending key for api.wandb.ai to your netrc file: /root/.netrc
wandb: Currently logged in as: nishkala-aistuff (nishkala-aistuff-georgia-institute-of-technology) to https://api.wandb.ai. Use `wandb login --relogin` to force relogin


True

In [5]:
PROJECT_ROOT = "/content/drive/MyDrive/Colab_Notebooks/LLMs/ARIA_Lite_v2"

BENCHMARK_PATH = (
    f"{PROJECT_ROOT}/eval/llm_only/benchmarks/llm_only_v1.json"
)

RUNS_DIR = (
    f"{PROJECT_ROOT}/eval/llm_only/runs/prompt_comparison"
)

PROMPT_STATS_PATH = (
    f"{PROJECT_ROOT}/eval/llm_only/scores/prompt_augmentation_stats.csv"
)

OUTPUT_CSV = (
    f"{PROJECT_ROOT}/eval/llm_only/scores/prompt_judge_scores_v1.csv"
)

LEADERBOARD_CSV = (
    f"{PROJECT_ROOT}/eval/llm_only/scores/prompt_leaderboard_v1.csv"
)

JUDGE_MODEL = "Qwen/Qwen2.5-14B-Instruct"

WANDB_PROJECT = "ARIA-Lite-Eval"
WANDB_GROUP = "prompt_augmentation_judge_v1"

In [6]:
with open(BENCHMARK_PATH, "r") as f:
    benchmark = json.load(f)

benchmark_lookup = {
    item["query_id"]: item
    for item in benchmark
}

print("Loaded benchmark:", len(benchmark_lookup))

Loaded benchmark: 4


In [7]:
stats_df = pd.read_csv(
    PROMPT_STATS_PATH
)

print(stats_df.shape)

stats_df.head()

(16, 4)


,query_id,prompt_name,latency_sec,response_tokens
0,Q1,prompt_v1,51.893861,254
1,Q2,prompt_v1,37.378967,300
2,Q3,prompt_v1,17.406169,149
3,Q4,prompt_v1,24.901402,210
4,Q1,prompt_v2_concise,18.553724,160


In [8]:
bnb_config = BitsAndBytesConfig(
    load_in_4bit=True,
    bnb_4bit_compute_dtype=torch.float16,
    bnb_4bit_quant_type="nf4",
    bnb_4bit_use_double_quant=True
)

tokenizer = AutoTokenizer.from_pretrained(
    JUDGE_MODEL
)

model = AutoModelForCausalLM.from_pretrained(
    JUDGE_MODEL,
    quantization_config=bnb_config,
    device_map="auto"
)

model.eval()

print("Judge loaded.")

/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:112: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


config.json:   0%|          | 0.00/663 [00:00<?, ?B/s]

tokenizer_config.json:   0%|          | 0.00/7.30k [00:00<?, ?B/s]

vocab.json:   0%|          | 0.00/2.78M [00:00<?, ?B/s]

merges.txt:   0%|          | 0.00/1.67M [00:00<?, ?B/s]

...

Fetching 8 files:   0%|          | 0/8 [00:00<?, ?it/s]

Loading weights:   0%|          | 0/579 [00:00<?, ?it/s]

/usr/local/lib/python3.12/dist-packages/bitsandbytes/backends/cuda/ops.py:213: FutureWarning: _check_is_size will be removed in a future PyTorch release along with guard_size_oblivious.     Use _check(i >= 0) instead.
  torch._check_is_size(blocksize)


generation_config.json:   0%|          | 0.00/242 [00:00<?, ?B/s]

Judge loaded.


In [9]:
def build_judge_prompt(
    query,
    sections,
    reference_answer,
    expected_topics,
    candidate_answer
):

    return f"""
You are an expert biomedical evaluator.

Evaluate the candidate answer against:

1. The provided evidence sections
2. The reference answer
3. Expected topics

Do NOT reward verbosity.

Reward only:

- factual correctness
- completeness
- grounding in evidence
- biomedical validity

Score each category from 0 to 5.

Return ONLY:

CORRECTNESS: X
COMPLETENESS: X
GROUNDING: X
BIOMEDICAL_SOUNDNESS: X

REASONING:
<brief explanation>

--------------------------------

QUERY:
{query}

--------------------------------

SECTIONS:
{sections}

--------------------------------

REFERENCE ANSWER:
{reference_answer}

--------------------------------

EXPECTED TOPICS:
{expected_topics}

--------------------------------

CANDIDATE ANSWER:
{candidate_answer}
"""

In [10]:
def judge_answer(prompt):

    messages = [
        {
            "role": "user",
            "content": prompt
        }
    ]

    text = tokenizer.apply_chat_template(
        messages,
        tokenize=False,
        add_generation_prompt=True
    )

    inputs = tokenizer(
        text,
        return_tensors="pt",
        truncation=True,
        max_length=12000
    ).to(model.device)

    outputs = model.generate(
        **inputs,
        max_new_tokens=250,
        do_sample=False,
        temperature=0.0
    )

    generated = outputs[0][
        inputs["input_ids"].shape[1]:
    ]

    return tokenizer.decode(
        generated,
        skip_special_tokens=True
    )

In [11]:
def extract_score(text, key):

    pattern = rf"{key}\s*:\s*(\d+)"

    match = re.search(
        pattern,
        text,
        flags=re.IGNORECASE
    )

    if match:
        return int(match.group(1))

    return 0

In [12]:
wandb.init(
    project=WANDB_PROJECT,
    group=WANDB_GROUP,
    name="prompt_augmentation_judge"
)

In [13]:
results = []

prompt_dirs = sorted(
    [
        p for p in Path(RUNS_DIR).iterdir()
        if p.is_dir()
    ]
)

for prompt_dir in prompt_dirs:

    prompt_name = prompt_dir.name

    print("\n")
    print("="*80)
    print(prompt_name)
    print("="*80)

    for json_file in prompt_dir.glob("*.json"):

        with open(json_file, "r") as f:
            run_data = json.load(f)

        query_id = run_data["query_id"]

        benchmark_item = benchmark_lookup[
            query_id
        ]

        sections = "\n\n".join([
            s["text"]
            for s in benchmark_item["sections"]
        ])

        expected_topics = ", ".join(
            benchmark_item["expected_topics"]
        )

        prompt = build_judge_prompt(
            query=benchmark_item["query"],
            sections=sections,
            reference_answer=benchmark_item["reference_answer"],
            expected_topics=expected_topics,
            candidate_answer=run_data["response"]
        )

        judgment = judge_answer(prompt)

        correctness = extract_score(
            judgment,
            "CORRECTNESS"
        )

        completeness = extract_score(
            judgment,
            "COMPLETENESS"
        )

        grounding = extract_score(
            judgment,
            "GROUNDING"
        )

        biomedical = extract_score(
            judgment,
            "BIOMEDICAL_SOUNDNESS"
        )

        total = (
            correctness
            + completeness
            + grounding
            + biomedical
        )

        row = {
            "query_id": query_id,
            "prompt_name": prompt_name,
            "correctness": correctness,
            "completeness": completeness,
            "grounding": grounding,
            "biomedical_soundness": biomedical,
            "total": total,
            "judge_output": judgment
        }

        results.append(row)

        wandb.log(row)

        print(
            f"{query_id} -> {total}/20"
        )



prompt_v1


[transformers] The following generation flags are not valid and may be ignored: ['temperature']. Set `TRANSFORMERS_VERBOSITY=info` for more details.
/usr/local/lib/python3.12/dist-packages/bitsandbytes/backends/cuda/ops.py:468: FutureWarning: _check_is_size will be removed in a future PyTorch release along with guard_size_oblivious.     Use _check(i >= 0) instead.
  torch._check_is_size(blocksize)


Q1 -> 16/20
Q2 -> 16/20
Q3 -> 20/20
Q4 -> 16/20


prompt_v2_concise
Q1 -> 16/20
Q2 -> 16/20
Q3 -> 20/20
Q4 -> 17/20


prompt_v3_grounded
Q1 -> 16/20
Q2 -> 17/20
Q3 -> 20/20
Q4 -> 17/20


prompt_v4_rubric
Q1 -> 15/20
Q2 -> 15/20
Q3 -> 11/20
Q4 -> 15/20


In [14]:
scores_df = pd.DataFrame(
    results
)

scores_df.head()

,query_id,prompt_name,correctness,completeness,grounding,biomedical_soundness,total,judge_output
0,Q1,prompt_v1,4,4,4,4,16,CORRECTNESS: 4\nCOMPLETENESS: 4\nGROUNDING: 4\...
1,Q2,prompt_v1,4,4,4,4,16,CORRECTNESS: 4\nCOMPLETENESS: 4\nGROUNDING: 4\...
2,Q3,prompt_v1,5,5,5,5,20,CORRECTNESS: 5\nCOMPLETENESS: 5\nGROUNDING: 5\...
3,Q4,prompt_v1,4,4,4,4,16,CORRECTNESS: 4\nCOMPLETENESS: 4\nGROUNDING: 4\...
4,Q1,prompt_v2_concise,4,4,4,4,16,CORRECTNESS: 4\nCOMPLETENESS: 4\nGROUNDING: 4\...


In [15]:
final_df = scores_df.merge(
    stats_df,
    on=[
        "query_id",
        "prompt_name"
    ],
    how="left"
)

final_df.head()

,query_id,prompt_name,correctness,completeness,grounding,biomedical_soundness,total,judge_output,latency_sec,response_tokens
0,Q1,prompt_v1,4,4,4,4,16,CORRECTNESS: 4\nCOMPLETENESS: 4\nGROUNDING: 4\...,51.893861,254
1,Q2,prompt_v1,4,4,4,4,16,CORRECTNESS: 4\nCOMPLETENESS: 4\nGROUNDING: 4\...,37.378967,300
2,Q3,prompt_v1,5,5,5,5,20,CORRECTNESS: 5\nCOMPLETENESS: 5\nGROUNDING: 5\...,17.406169,149
3,Q4,prompt_v1,4,4,4,4,16,CORRECTNESS: 4\nCOMPLETENESS: 4\nGROUNDING: 4\...,24.901402,210
4,Q1,prompt_v2_concise,4,4,4,4,16,CORRECTNESS: 4\nCOMPLETENESS: 4\nGROUNDING: 4\...,18.553724,160


In [16]:
final_df.to_csv(
    OUTPUT_CSV,
    index=False
)

print(
    "Saved:",
    OUTPUT_CSV
)

Saved: /content/drive/MyDrive/Colab_Notebooks/LLMs/ARIA_Lite_v2/eval/llm_only/scores/prompt_judge_scores_v1.csv


In [17]:
leaderboard = (
    final_df
    .groupby("prompt_name")
    .agg({
        "correctness":"mean",
        "completeness":"mean",
        "grounding":"mean",
        "biomedical_soundness":"mean",
        "total":"mean",
        "response_tokens":"mean",
        "latency_sec":"mean"
    })
    .round(3)
    .sort_values(
        "total",
        ascending=False
    )
)

leaderboard

,correctness,completeness,grounding,biomedical_soundness,total,response_tokens,latency_sec
prompt_name,,,,,,,
prompt_v3_grounded,4.25,4.25,4.25,4.75,17.50,210.75,24.998
prompt_v2_concise,4.25,4.25,4.25,4.50,17.25,150.50,19.728
prompt_v1,4.25,4.25,4.25,4.25,17.00,228.25,32.895
prompt_v4_rubric,3.75,2.75,3.75,3.75,14.00,300.00,37.357


In [18]:
leaderboard["score_per_token"] = (
    leaderboard["total"]
    / leaderboard["response_tokens"]
)

leaderboard = leaderboard.sort_values(
    "total",
    ascending=False
)

leaderboard

,correctness,completeness,grounding,biomedical_soundness,total,response_tokens,latency_sec,score_per_token
prompt_name,,,,,,,,
prompt_v3_grounded,4.25,4.25,4.25,4.75,17.50,210.75,24.998,0.083037
prompt_v2_concise,4.25,4.25,4.25,4.50,17.25,150.50,19.728,0.114618
prompt_v1,4.25,4.25,4.25,4.25,17.00,228.25,32.895,0.074480
prompt_v4_rubric,3.75,2.75,3.75,3.75,14.00,300.00,37.357,0.046667


In [19]:
leaderboard.to_csv(
    LEADERBOARD_CSV
)

print(
    "Saved:",
    LEADERBOARD_CSV
)

Saved: /content/drive/MyDrive/Colab_Notebooks/LLMs/ARIA_Lite_v2/eval/llm_only/scores/prompt_leaderboard_v1.csv


In [20]:
wandb.log({
    "prompt_leaderboard":
    wandb.Table(
        dataframe=leaderboard.reset_index()
    )
})

wandb.log({
    "prompt_scores":
    wandb.Table(
        dataframe=final_df
    )
})

wandb.finish()

biomedical_soundness,▅▅█▅▅▅██▅███▅▅▁▅
completeness,▆▆█▆▆▆█▆▆▆█▆▃▃▁▃
correctness,▅▅█▅▅▅█▅▅▅█▅▅▅▁▅
grounding,▅▅█▅▅▅█▅▅▅█▅▅▅▁▅
total,▅▅█▅▅▅█▆▅▆█▆▄▄▁▄
biomedical_soundness,4
completeness,3
correctness,4
grounding,4
judge_output,CORRECTNESS: 4COMPL...
prompt_name,prompt_v4_rubric
